# Solutions · 05 · Numerical integration

Worked solutions to the exercises of [notebook 05](../notebooks/05_integration.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import integrate
from scipy.integrate import quad
def power(t):
    return 40 + 25*np.sin(np.pi*t/8)**2 + 30*np.exp(-(t-0.5)**2/0.08) - 20*np.exp(-(t-4)**2/0.1)
E_exact = quad(power, 0, 8, epsabs=1e-13, limit=200)[0]

## Exercise 1

How often must the power be logged for the trapezoidal energy estimate to be within 0.5 %?

In [2]:
def error_for(minutes, offset):
    # logging every `minutes`, first sample `offset` minutes after the start (plus the start and end)
    t = np.unique(np.r_[0.0, np.arange(offset/60, 8, minutes/60), 8.0])
    return integrate.trapezoid(power(t), t) / E_exact - 1

for minutes in (60, 30, 20, 15, 10, 5):
    errs = [error_for(minutes, off) for off in np.arange(0, minutes, 1.0)]
    worst = max(errs, key=abs)
    print(f"every {minutes:>2} min: error {errs[0]*100:+.3f} % on the hour grid, worst case over all "
          f"schedule offsets {worst*100:+.3f} %{'   <- always within 0.5 %' if abs(worst) < 0.005 else ''}")

every 60 min: error -5.136 % on the hour grid, worst case over all schedule offsets -5.148 %
every 30 min: error +0.144 % on the hour grid, worst case over all schedule offsets -0.568 %
every 20 min: error -0.037 % on the hour grid, worst case over all schedule offsets -0.116 %   <- always within 0.5 %
every 15 min: error -0.018 % on the hour grid, worst case over all schedule offsets -0.053 %   <- always within 0.5 %
every 10 min: error -0.009 % on the hour grid, worst case over all schedule offsets -0.018 %   <- always within 0.5 %
every  5 min: error -0.002 % on the hour grid, worst case over all schedule offsets -0.003 %   <- always within 0.5 %


On the regular grid starting at the full hour, 30-minute logging happens to pass - but only because one
sample falls exactly on the peak of the start-up surge at $t$ = 0.5 h. Shift the schedule and the error
exceeds 0.5 %. The honest answer looks at the **worst case over all schedule offsets**: logging every
20 minutes or faster is needed. Choose a logging interval from the fastest event you need to capture,
not from the integration formula.

## Exercise 2

Show that Simpson's rule on *equally* spaced points is exact for cubics, but not on unequal
   spacing. (Integrate $t^3$ on both kinds of grid.)

In [3]:
t_eq = np.linspace(0, 3, 11)
rng = np.random.default_rng(0)
t_uneven = np.sort(np.r_[0, rng.uniform(0, 3, 9), 3])
exact = 3**4 / 4
print(f"equal spacing:   Simpson error for t^3 = {integrate.simpson(t_eq**3, t_eq) - exact:+.2e}")
print(f"unequal spacing: Simpson error for t^3 = {integrate.simpson(t_uneven**3, t_uneven) - exact:+.2e}")
print(f"unequal spacing: Simpson error for t^2 = {integrate.simpson(t_uneven**2, t_uneven) - 3**3/3:+.2e}")

equal spacing:   Simpson error for t^3 = +0.00e+00
unequal spacing: Simpson error for t^3 = +3.76e-02
unequal spacing: Simpson error for t^2 = +0.00e+00


On equal spacing Simpson's rule integrates cubics exactly: the parabola's error for the cubic term is an
odd function about the middle point and cancels. With unequal spacing the symmetry is lost and a small
error remains, while quadratics are still exact (the parabola *is* the function).

## Exercise 3

The logged power has random measurement noise of ±1 kW (standard deviation). Use Monte Carlo to
   find the resulting uncertainty in the energy. Is it larger or smaller than the sampling error?

In [4]:
t_log = np.arange(0, 8.01, 0.5)
P_log = power(t_log)
rng = np.random.default_rng(5)
E_noisy = np.array([integrate.trapezoid(P_log + rng.normal(0, 1.0, P_log.size), t_log) for _ in range(20000)])
w = np.full(t_log.size, 0.5); w[[0, -1]] = 0.25                    # trapezoid weights (h = 0.5 h)
print(f"noise-induced uncertainty: Monte Carlo {E_noisy.std():.3f} kWh, formula sigma*sqrt(sum w²) = {np.sqrt(np.sum(w**2)):.3f} kWh")
print(f"sampling (discretisation) error of the same data: {integrate.trapezoid(P_log, t_log) - E_exact:+.3f} kWh")

noise-induced uncertainty: Monte Carlo 1.959 kWh, formula sigma*sqrt(sum w²) = 1.969 kWh
sampling (discretisation) error of the same data: +0.610 kWh


The random noise contributes an uncertainty of about 2 kWh - **larger** than the error from sampling only
every 30 minutes. Because the energy is a weighted sum of readings, the Monte Carlo result can be
checked exactly: $\sigma_E = \sigma\sqrt{\sum w_i^2}$. Averaging over many readings reduces the effect of
noise, but it never disappears; a faster logger would reduce both errors.

## Exercise 4

**Implement it yourself:** Boole's rule applies the weights $\tfrac{2h}{45}(7, 32, 12, 32, 7)$ to groups of four intervals. Implement it, measure its order on the power profile (theory: 6), and explain why it still cannot capture the start-up surge from 30-minute data.

In [5]:
def boole(f, a, b, n):
    if n % 4:
        raise ValueError("n must be a multiple of 4")
    x = np.linspace(a, b, n + 1); y = f(x); h = (b - a) / n
    w = np.tile([14, 32, 12, 32], n // 4).astype(float); w = np.r_[w, 7.0]; w[0] = 7.0
    return 2*h/45 * np.sum(w * y)

ns = np.array([16, 32, 64, 128, 256, 512, 1024])
errs = np.array([abs(boole(power, 0, 8, n) - E_exact) for n in ns])
print("error reduction per halving of h:", np.round(errs[:-1] / errs[1:], 1), " (order 6 means 2^6 = 64)")
print(f"order fitted on the finest grids: {np.polyfit(np.log(8/ns[-3:]), np.log(errs[-3:]), 1)[0]:.2f}")
print(f"30-minute data (n = 16): Boole error {boole(power, 0, 8, 16) - E_exact:+.2f} kWh, "
      f"trapezoid {integrate.composite(power, 0, 8, 16, 'trapezoid') - E_exact:+.2f} kWh")

error reduction per halving of h: [  11.1   47.2 1974.9   56.9   61.6   63.6]  (order 6 means 2^6 = 64)
order fitted on the finest grids: 5.97
30-minute data (n = 16): Boole error +9.60 kWh, trapezoid +0.61 kWh


Where two groups of four intervals meet, their end weights (7 + 7) add to 14 - hence the repeating
pattern 14, 32, 12, 32. On fine grids the error falls by the factor 64 per halving that order 6 predicts.
On coarse grids it behaves irregularly - improving by factors of about 11 and 47, then by almost 2000 at
once as the grid starts to resolve the surge: the *pre-asymptotic* regime, where order-of-accuracy
statements do not yet apply. On the 30-minute data Boole's rule is far **worse** than the trapezoid: it fits
high-degree polynomials through the samples, which overshoot around an event narrower than the sample
spacing. More accuracy requires more data, not a cleverer formula.